<a href="https://colab.research.google.com/github/ysuter/FHNW-BSUD-Part2/blob/main/L10_TextMining/in_context_learning_qwen_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Context Learning with Free Open Models (Qwen 2.5 Instruct)

In this lab you will:

- Understand **in-context learning** (ICL)
- Compare **zero-shot** vs **few-shot** prompting
- Use a free, open-source LLM (**Qwen2.5-7B-Instruct**) for:
  - Sentiment classification
  - Style transfer
  - JSON-style structured extraction

We run everything locally in this Colab using Hugging Face Transformers — no paid API key needed.


## 1. Install required libraries

In [ ]:
!pip install -q transformers accelerate sentencepiece bitsandbytes

## 2. Load a Qwen model (free)

We will use **Qwen/Qwen2.5-7B-Instruct** from Hugging Face.

> ⚠️ Tip: A GPU runtime is strongly recommended (Colab: *Runtime → Change runtime type → GPU*).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "Qwen/Qwen2.5-7B-Instruct"

print("Loading tokenizer and model… This may take a minute.")
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",          # automatically choose GPU if available
    dtype=torch.float16,  # use half-precision to save memory
    load_in_4bit=True           # quantize to 4-bit (bitsandbytes)
)

print("Model loaded:", model_id)

## 3. Helper function for text generation

We wrap model inference in a small `generate()` helper for convenience.

In [ ]:
def generate(prompt, max_new_tokens=256, temperature=0.3):
    """Generate text from the Qwen model for a given prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)


# Quick sanity check
test_output = generate("In one or two sentences, explain what in-context learning is.")
print(test_output)

## 4. Zero-shot vs few-shot sentiment classification

We start with a simple sentiment classification task:
- **Zero-shot:** only instructions
- **Few-shot:** instructions **plus examples** in the prompt


### 4.1 Create a small dataset

In [ ]:
reviews = [
    "The product arrived 3 days late and the quality is terrible.",
    "Absolutely fantastic service, I will definitely order again!",
    "It's okay, not great but not awful either.",
    "The battery life is amazing and the camera is super sharp.",
    "Customer support never replied to my emails."
]

for i, r in enumerate(reviews, start=1):
    print(f"{i}. {r}")

### 4.2 Zero-shot classification

In [ ]:
prompt_zero = """You are a sentiment classifier.

Classify the sentiment of each customer review as **positive**, **negative**, or **neutral**.

Justify briefly in 1 short sentence per review.

Reviews:
"""

for i, r in enumerate(reviews, start=1):
    prompt_zero += f"Review {i}: {r}\n"

print("=== Zero-shot prompt ===")
print(prompt_zero)
print("\n=== Model output ===")
print(generate(prompt_zero, max_new_tokens=256))

> **Observation task:**  
> - Does the model always use only *positive/negative/neutral*?  
> - How consistent is the structure of the answers?

### 4.3 Few-shot classification (in-context examples)

In [ ]:
prompt_few = """You are a sentiment classifier.

You must output exactly one line per review in the format:
Review X: <sentiment> - <very short explanation>

Use one of these labels only: positive, negative, neutral.

Here are some examples:

Review: "This laptop is super fast and light."
Output: Review 0: positive - performance is very good

Review: "The headphones broke after one day."
Output: Review 0: negative - broke quickly

Review: "The phone works as expected."
Output: Review 0: neutral - nothing especially good or bad

Now classify the following reviews:

"""

for i, r in enumerate(reviews, start=1):
    prompt_few += f'Review {i}: "{r}"\n'

print("=== Few-shot prompt ===")
print(prompt_few)
print("\n=== Model output ===")
print(generate(prompt_few, max_new_tokens=256))

#### Reflection

- Did the outputs follow the **format of the examples** more closely?
- Are the labels and explanations more standardized than in the zero-shot case?

## 5. Style transfer with and without examples

We now see how in-context examples can control **writing style** (e.g. informal → professional).

### 5.1 Zero-shot style transfer

In [ ]:
raw_email = """hey team,
i can't finish the report by tomorrow. i'm completely overloaded.
can someone take over some parts? thx
"""

prompt_zero_style = f"""Rewrite the following email in a professional and polite tone.
Keep the meaning, but improve clarity and politeness.

Email:
{raw_email}
"""

print("=== Zero-shot style prompt ===")
print(prompt_zero_style)
print("\n=== Model output ===")
print(generate(prompt_zero_style, max_new_tokens=256))

### 5.2 Few-shot style transfer with an example

In [ ]:
raw_email_2 = """hey boss,
i'll be late to the meeting again, traffic is crazy.
just start without me.
"""

prompt_few_style = f"""You transform informal emails into polite, professional business emails.

Example:
Informal:
"hey anna, i can't join the call, my laptop died lol"

Professional:
"Hi Anna,

Unfortunately, I will not be able to join the call because my laptop suddenly stopped working.
I will review the meeting notes afterwards and follow up on any action items assigned to me.

Best regards,
[Your Name]"

Now transform the following email.

Informal:
{raw_email_2}

Professional:
"""

print("=== Few-shot style prompt ===")
print(prompt_few_style)
print("\n=== Model output ===")
print(generate(prompt_few_style, max_new_tokens=256))

#### Exercise (style design)

1. Pick a target style, e.g.
   - Very short and friendly (may include emojis)
   - Very formal and legalistic
   - Like a supportive coach

2. Create your own prompt with:
   - A clear instruction.
   - 1–2 example pairs (informal → target style).

3. Test your prompt on 3 new emails you invent.  
4. Observe: Does the model follow your style consistently?

## 6. In-context learning for structured JSON-style outputs

We now ask the model to output **structured data** (JSON-like) and see how examples improve consistency.

### 6.1 Zero-shot JSON extraction

In [ ]:
text = """Hi, my name is Jane Doe. You can reach me at jane.doe@example.com
or on my mobile +41 79 123 45 67. I'm interested in the Business AI course
starting in March 2027.
"""

prompt_zero_json = f"""Extract the following information from the text:
- full_name
- email
- phone
- course_name
- start_date (as an ISO date if you can, e.g. 2026-03-01)

Return the result as JSON.

Text:
{text}
"""

print("=== Zero-shot JSON prompt ===")
print(prompt_zero_json)
print("\n=== Model output ===")
print(generate(prompt_zero_json, max_new_tokens=256))

### 6.2 Few-shot JSON extraction with schema + example

In [ ]:
prompt_few_json = f"""You extract structured information as JSON.

Use this JSON schema exactly:
{{
  "full_name": "",
  "email": "",
  "phone": "",
  "course_name": "",
  "start_date": ""
}}

Example:

Text:
"I am Max Mustermann. Contact me at max@example.com or +49 151 234567.
I'm interested in the Data Science Bootcamp starting in September 2025."

JSON:
{{
  "full_name": "Max Mustermann",
  "email": "max@example.com",
  "phone": "+49 151 234567",
  "course_name": "Data Science Bootcamp",
  "start_date": "2025-09-01"
}}

Now process this text:

Text:
{text}

Return ONLY JSON with valid syntax.
"""

print("=== Few-shot JSON prompt ===")
print(prompt_few_json)
few_json_output = generate(prompt_few_json, max_new_tokens=256)
print("\n=== Model output ===")
print(few_json_output)

# Optional: try to parse as JSON
import json
try:
    parsed = json.loads(few_json_output)
    print("\nParsed as Python dict:")
    print(parsed)
except json.JSONDecodeError as e:
    print("\nWarning: Model output is not valid JSON:", e)

#### Exercise (robust extraction)

1. Create 3–4 **noisy texts** with missing fields, multiple phone numbers, etc.
2. Extend your prompt with explicit rules, for example:
   - "If a field is missing, leave it as an empty string."
3. Add or adjust examples to show how to handle missing data.
4. Run the model and check:
   - Are outputs still valid JSON?
   - Does the model follow your rules?

## 7. Bonus: In-context learning for Dish / Ingredient tagging

In this bonus example, we use **in-context learning** for a more structured NLP task:

> Given a menu item description, identify **Dishes** vs **Ingredients**.

We provide several labeled examples in the prompt and then test the model on new descriptions.

The model must continue the pattern:
- `Target types: Dish; Ingredient`
- Repeated blocks of `Text: ...` and `Entities: ...`

This is a realistic scenario from information extraction in the food/restaurant domain.


In [ ]:
# Base few-shot prompt with multiple labeled examples
base_input = """Target types: Dish; Ingredient
Text: French Fries with Ketchup, Fresh & Deluxe Sides, Fresh Sides, Fresh Sides a la Carte 3, Brunch
Entities: French Fries is Dish, Ketchup is Ingredient
Text: The Spicy Pig Tavern Crust Pizza with Thin Sliced Ham, Smoked Bacon, Pineapple, Jalapenos & BJs Signature 5 Cheese Blend, BJs Tavern Crust Pizzas, Tavern Crust, Pizza
Entities: Pizza is Dish, Thin Sliced Ham is Ingredient, Smoked Bacon is Ingredient, Pineapple is Ingredient, Jalapenos is Ingredient, Cheese is Ingredient
Text: Turtle Caramel Nut Milkshake w/ Whipped Cream & Candy Pieces, Milkshakes, Specialty Milkshakes, Hand Dipped Milkshakes, Regular
Entities: Milkshake is Dish, Caramel is Ingredient, Nut is Ingredient, Whipped Cream is Ingredient, Candy Pieces is Ingredient
Text: Turkey Breast on 9 Grain Wheat Bread w/ Cucumbers, Green Peppers, Lettuce, Red Onions & Tomatoes w/out Cheese & Sauce, All Sandwiches, Fresh Fit Choices, Sub of the Day, Footlong
Entities: Turkey Breast is Dish, 9 Grain Wheat Bread is Ingredient, Cucumbers is Ingredient, Green Peppers is Ingredient, Lettuce is Ingredient, Red Onions is Ingredient, Tomatoes is Ingredient
Text: {0}
Entities: """

# A few test cases with expected labels
test_cases = []
test_cases.append((
    "French Fries | Noodles w/ Whole Milk & Espresso, Medium, McCafe",
    "Cappuccino is Dish, Espresso is Ingredient, Whole Milk is Ingredient"
))
test_cases.append((
    "McCafe French Vanilla Cappuccino w/ Whole Milk & Espresso, Medium, McCafe",
    "Cappuccino is Dish, Espresso is Ingredient, Whole Milk is Ingredient"
))
test_cases.append((
    "Fried Shrimp w/ Cocktail Sauce, Shrimp Your Way, Shrimp & Classic Combinations",
    "Fried Shrimp is Dish, Cocktail Sauce is Ingredient"
))
test_cases.append((
    "Mojito w/ Bacardi Superior Rum, Fresh Squeezed Lime, Mint, Pure Cane Sugar & Club Soda, Signature Cocktails, Drinks, Alcoholic Beverages",
    "Mojito is Dish, Bacardi Superior Rum is Ingredient, Fresh Squeezed Lime is Ingredient, Mint is Ingredient, Pure Cane Sugar is Ingredient, Club Soda is Ingredient"
))
test_cases.append((
    "Jumbo Coconut Shrimp, for 4 Course Feast w/ Pina Colada Sauce",
    "Jumbo Coconut Shrimp is Dish, Pina Colada Sauce is Ingredient"
))
test_cases.append((
    "Snow Crab Legs, Add An Additional Cluster, Dinner Entrees",
    "Snow Crab Legs is Dish"
))


def extract_last_entities_block(full_generated_text: str) -> str:
    """Heuristic: return the text after the final 'Entities:' marker."""
    marker = "Entities:"
    if marker not in full_generated_text:
        return full_generated_text.strip()
    return full_generated_text.split(marker)[-1].strip()


print("Running dish/ingredient tagging with in-context learning...\n")

for n, (description, expected) in enumerate(test_cases, start=1):
    cur_input = base_input.format(description)
    out_text = generate(cur_input, max_new_tokens=80)
    actual_out = extract_last_entities_block(out_text)

    print(f"=== Test case {n} ===")
    print("Description: ", description)
    print("Expected:   ", expected)
    print("Predicted:  ", actual_out)
    print()

### Discussion

- How closely does the model follow the **pattern** from the few-shot examples?
- Does it correctly separate **Dish** vs **Ingredient**?
- Where does it make mistakes (e.g., missing an ingredient, or misclassifying a component)?

This is a good example of how **task-specific labeling schemes** can often be implemented
with *only* prompt design and in-context examples — no additional training required.
